# 📈 O-ISAC Trend Analysis (V4 Preview)

This notebook analyzes the **Unified V4 Dataset** to visualize key performance trade-offs in Optical Integrated Sensing and Communication.

### Objectives (PRISMA Section 11.2)
1. **Trade-off Analysis:** Visualize relationships between Communication Rate, Sensing Resolution, and Range.
2. **Scenario Distribution:** Understand the landscape of Fiber vs. FSO vs. VLC research.
3. **Technology Clustering:** Identify operating regimes for different optical technologies.

---

In [ ]:
# @title 1. Setup & Load Data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Load Data
csv_path = "../../data/ext_v4_uni.csv"
try:
    df = pd.read_csv(csv_path)
    print(f"✅ Loaded dataset with {len(df)} papers.")
except FileNotFoundError:
    print("❌ CSV not found! Please run 'synthesis_v4_unified.py' first.")

# Preview
df.head()

## 🧹 2. Data Cleaning
Convert 'NR' (Not Reported) to NaN and ensure numeric columns are actually numeric.

In [ ]:
# Convert NR to NaN
import numpy as np
df_clean = df.replace("NR", np.nan)

# Convert columns to numeric
numeric_cols = ["Bitrate_Gbps", "Resolution_m", "Distance_m", "Wavelength_nm"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Filter for rows with at least some numeric data
df_viz = df_clean.dropna(subset=numeric_cols, how='all').copy()
print(f"📊 Papers with at least one numeric metric: {len(df_viz)}")

# Simplified Scenario Labels for Plotting
def simplify_scenario(s):
    s = str(s).lower()
    if "fiber" in s or "fibre" in s or "cabled" in s: return "Fiber-ISAC"
    if "fso" in s or "free-space" in s: return "FSO-ISAC"
    if "vlc" in s or "visible" in s: return "VLC-ISAC"
    if "lidar" in s: return "LiDAR-ISAC"
    return "Hybrid/Other"

df_viz['Scenario_Group'] = df_viz['Scenario'].apply(simplify_scenario)
df_viz['Scenario_Group'].value_counts()

## 🚀 3. Visualization: Rate vs. Range vs. Resolution

This 3D scatter plot (projected to 2D with color/size) shows the "Golden Triangle" of O-ISAC performance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Filter for valid X and Y
plot_data = df_viz.dropna(subset=["Distance_m", "Bitrate_Gbps"])

sns.scatterplot(
    data=plot_data,
    x="Distance_m",
    y="Bitrate_Gbps",
    hue="Scenario_Group",
    style="Scenario_Group",
    size="Resolution_m",
    sizes=(20, 200),
    alpha=0.7,
    palette="viridis",
    ax=ax
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("O-ISAC Landscape: Bitrate vs. Range", fontsize=16)
ax.set_xlabel("Comm/Sensing Distance (m) [Log Scale]", fontsize=12)
ax.set_ylabel("Data Rate (Gbps) [Log Scale]", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

# Annotate top performers
for i, row in plot_data.nlargest(3, "Bitrate_Gbps").iterrows():
    ax.text(row["Distance_m"], row["Bitrate_Gbps"], f" {row['Paper_ID']}", fontsize=9)

plt.tight_layout()
plt.show()

## 🎯 4. Visualization: Sensing Resolution vs. Bitrate
Does higher speed mean lower accuracy? (The Capacity-Accuracy Trade-off)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

plot_data_res = df_viz.dropna(subset=["Bitrate_Gbps", "Resolution_m"])

sns.regplot(
    data=plot_data_res,
    x="Bitrate_Gbps",
    y="Resolution_m",
    logx=True,
    scatter_kws={'alpha':0.5},
    line_kws={'color': 'red'},
    ax=ax
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Trade-off: Bitrate vs. Sensing Resolution", fontsize=16)
ax.set_xlabel("Data Rate (Gbps)", fontsize=12)
ax.set_ylabel("Resolution (m) [Lower is Better]", fontsize=12)
ax.invert_yaxis() # Invert Y so top-right is 'High Speed, High Accuracy' (Best)

plt.show()